In [3]:
import cv2
import numpy as np

def create_sample_receipt(output_path="receipt_sample.jpg"):
    # 1. 흰색 배경의 영수증 이미지 생성 (높이 600, 너비 400, 3채널 RGB)
    canvas = np.ones((600, 400, 3), dtype=np.uint8) * 255

    # 2. 영수증에 들어갈 텍스트 내용 정의
    lines = [
        "================================",
        "       [ EASY OCR STORE ]       ",
        "================================",
        "사업자번호: 123-45-67890",
        "TEL: 02-1234-5678",
        "주소: 서울시 강남구 테헤란로 123",
        "--------------------------------",
        "품명            수량        금액",
        "--------------------------------",
        "Python Book       1       30,000",
        "Coffee            2       10,000",
        "--------------------------------",
        "합계 금액:                40,000",
        "================================",
        "2026-08-27 14:30:00",
        "감사합니다."
    ]

    # 3. 이미지에 텍스트 그리기 (OpenCV 기본 폰트 적용)
    y_offset = 40
    for line in lines:
        # cv2.putText는 한글 출력이 깨질 수 있으므로 영문/숫자 중심으로 작성하거나, 
        # 한글 출력을 원할 경우 PIL(ImageDraw)을 활용하는 것이 좋습니다.
        cv2.putText(
            canvas, line, (20, y_offset), 
            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA
        )
        y_offset += 32

    # 4. 이미지 저장
    cv2.imwrite(output_path, canvas)
    print(f"샘플 영수증 이미지가 성공적으로 생성되었습니다: {output_path}")

if __name__ == "__main__":
    create_sample_receipt()

샘플 영수증 이미지가 성공적으로 생성되었습니다: receipt_sample.jpg


In [4]:
# 파이썬에서 Tesseract OCR 엔진을 사용할 수 있도록 연결해 주는 pytesseract 라이브러리를 불러옵니다.
import pytesseract

# 파이썬의 표준 이미지 처리 라이브러리인 Pillow(PIL)에서 이미지를 열고 다루기 위한 Image 모듈을 불러옵니다.
from PIL import Image

# [윈도우(Windows) 환경 전용 설정]
# pytesseract가 윈도우 시스템에 설치된 Tesseract 실행 파일(tesseract.exe)의 위치를 찾을 수 있도록 절대 경로를 지정합니다.
# (문자열 앞의 'r'은 역슬래시(\)를 이스케이프 문자가 아닌 일반 문자로 처리하는 원시 문자열(Raw String) 표기법입니다.)
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

# [macOS 환경 전용 설정 (주석 처리됨)]
# Apple Silicon(M1/M2/M3) Mac에서 Homebrew로 Tesseract를 설치했을 때의 실행 파일 경로입니다.
# macOS에서 실행할 경우 위의 윈도우 경로를 주석 처리하고 이 설정의 주석(#)을 해제하여 사용합니다.
# pytesseract.pytesseract.tesseract_cmd = '/opt/homebrew/bin/tesseract'

# 분석할 대상 이미지 파일의 이름(또는 상대/절대 경로)을 변수에 저장합니다.
image_path = 'receipt_sample.jpg'

# 1. Image.open(image_path): Pillow 라이브러리를 사용해 지정한 경로의 이미지 파일('receipt_sample.jpg')을 메모리에 로드합니다.
# 2. image_to_string(...): 로드된 이미지에서 문자를 감지하고 텍스트(String)로 변환하는 OCR을 수행합니다.
# 3. lang='kor+eng': 한국어('kor') 패키지와 영어('eng') 패키지를 동시에 사용하여 두 언어를 함께 인식하도록 설정합니다.
# 4. 추출된 최종 텍스트 결과는 문자열 형태로 변수 'text'에 저장됩니다.
text = pytesseract.image_to_string(Image.open(image_path), lang='kor+eng')

# 콘솔 화면에 Tesseract OCR 실행 결과의 시작을 알리는 구분선 타이틀을 출력합니다.
print("=== Tesseract 인식 결과 ===")

# pytesseract를 통해 이미지에서 최종적으로 추출된 텍스트 전체를 콘솔 화면에 출력합니다.
print(text)

=== Tesseract 인식 결과 ===
[EASY OCR STORE ]

사업자번호:123-45-67890

TEL: 02-1234-5678
주소:서울시 강남구테헤란로 123

PythonBook 1 30,000
Coffee 2 10,000

합계 금액:           40,000

2026-08-27 14:30:00
감사합니다.



In [5]:
# EasyOCR 라이브러리를 사용하기 위해 모듈을 불러옵니다.
import easyocr

# 1. EasyOCR의 핵심인 Reader 클래스의 인스턴스(객체)를 생성합니다.
# - ['ko', 'en']: 인식할 대상 언어로 한국어('ko')와 영어('en')를 지정합니다.
# - gpu=False: 연산에 GPU(CUDA)를 사용하지 않고 CPU로만 추론을 수행하도록 설정합니다. (GPU가 설치되어 있다면 True로 설정)
reader = easyocr.Reader(['ko', 'en'], gpu=False)

# 2. 지정한 이미지 파일('receipt_sample.jpg')에서 텍스트 감지 및 인식을 실행합니다.
# - readtext() 함수는 이미지 전처리, 글자 위치 탐지(Detection), 문자 인식(Recognition) 과정을 내부적으로 순차 수행합니다.
# - 인식 결과는 [(영역 좌표), '인식된 텍스트', 신뢰도 점수] 형태의 튜플(Tuple)들을 담은 리스트(List)로 반환됩니다.
results = reader.readtext('receipt_sample.jpg')

# 3. 콘솔 화면에 OCR 실행 결과를 알리는 구분선 타이틀을 출력합니다.
print("=== EasyOCR 인식 결과 ===")

# readtext()가 반환한 결과 리스트에서 하나씩 요소를 꺼내어 반복문을 실행합니다.
# - bbox: 글자가 감지된 영역의 네 꼭짓점 좌표 (Bounding Box)
# - text: 해당 영역에서 추출 및 디코딩된 문자열 (Text)
# - prob: 해당 문자가 정확하게 인식되었을 확률값 / 신뢰도 (Confidence Score, 0.0 ~ 1.0)
for bbox, text, prob in results:
    # 각 글자 영역의 추출 텍스트와 신뢰도(소수점 넷째 자리까지 표시)를 포맷팅하여 콘솔에 출력합니다.
    print(f"텍스트: {text} | 신뢰도: {prob:.4f}")

Using CPU. Note: This module is much faster with a GPU.


Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.1% Complete

c:\skn35\workspace\dl_workspace\.venv\Lib\site-packages\torch\ao\nn\quantized\dynamic\modules\rnn.py:162: UserWarning: torch.quantize_per_tensor, torch.quantize_per_channel and other quantized tensor creation functions that produce tensors with dtype torch.quint8, torch.qint8, and torch.qint32 are deprecated and will be removed in a future PyTorch release. Please see https://github.com/pytorch/pytorch/issues/184982 for more information. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\quantized\Quantizer.cpp:116.)
  w_ih = torch.quantize_per_tensor(
c:\skn35\workspace\dl_workspace\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


=== EasyOCR 인식 결과 ===
텍스트: [EASY OCRSTORE] | 신뢰도: 0.9708
텍스트: 사업자번호: 123-45-67890 | 신뢰도: 0.7563
텍스트: TEL:02-1234-5678 | 신뢰도: 0.7618
텍스트: 주소: 서울시 강남구 테혜란로 123 | 신뢰도: 0.8370
텍스트: 품명 | 신뢰도: 0.9955
텍스트: 수량 | 신뢰도: 0.9994
텍스트: 금액 | 신뢰도: 0.9998
텍스트: Python Book | 신뢰도: 0.7897
텍스트: 30,000 | 신뢰도: 0.9345
텍스트: Coffee | 신뢰도: 0.9676
텍스트: 10,000 | 신뢰도: 0.7165
텍스트: 합계 금액: | 신뢰도: 0.9312
텍스트: 40,000 | 신뢰도: 0.9801
텍스트: 2026-08-27 1430.00 | 신뢰도: 0.9172
텍스트: 감사합니다. | 신뢰도: 0.6642


In [6]:
# 이미지 생성 및 텍스트 렌더링을 위한 Pillow(PIL) 모듈을 불러옵니다.
from PIL import Image, ImageDraw, ImageFont


def create_business_registration_image(file_name="business_registration.jpg"):
    """
    OCR 파이프라인 테스트용 사업자등록증 형태의 샘플 이미지를 생성합니다.
    """
    # 1. 문서 규격 설정 (너비 600px, 높이 800px, 밝은 회색/아이보리 톤 배경)
    width, height = 600, 800
    image = Image.new("RGB", (width, height), color=(250, 250, 250))
    draw = ImageDraw.Draw(image)

    # 2. OS별 기본 폰트 설정 (한글 폰트 미설치 시 기본 폰트로 폴백)
    try:
        # Windows 환경 기준 맑은 고딕 폰트 사용
        font_title = ImageFont.truetype("malgunbd.ttf", 28)
        font_label = ImageFont.truetype("malgunbd.ttf", 16)
        font_content = ImageFont.truetype("malgun.ttf", 16)
    except Exception:
        # 폰트 로드 실패 시 예외 처리
        font_title = font_label = font_content = ImageFont.load_default()

    # 3. 문서 테두리 및 제목 영역 그리기
    draw.rectangle(
        [(20, 20), (width - 20, height - 20)], outline=(0, 0, 0), width=3
    )
    draw.rectangle(
        [(30, 30), (width - 30, height - 30)],
        outline=(100, 100, 100),
        width=1,
    )
    draw.text((180, 60), "사업자등록증", fill=(0, 0, 0), font=font_title)
    draw.line([(50, 110), (width - 50, 110)], fill=(0, 0, 0), width=2)

    # 4. 정규표현식 파싱을 위한 핵심 데이터 항목 정의
    # - 사업자등록번호 패턴: \d{3}-\d{2}-\d{5} (예: 123-45-67890)
    # - 전화번호 패턴: 0\d{1,2}-\d{3,4}-\d{4} (예: 02-1234-5678)
    items = [
        ("등록번호", "123-45-67890"),  # business_number 매칭 대상
        ("법인명(상호)", "(주)파이썬 인텔리전스"),
        ("대표자명", "홍길동"),
        ("개업연월일", "2024년 01월 01일"),
        ("사업장 소재지", "서울특별시 강남구 테헤란로 123, 4층"),
        ("대표 전화번호", "02-1234-5678"),  # phone_number 매칭 대상
        ("사업의 종류", "업태: 정보통신업 / 종목: 소프트웨어 개발"),
    ]

    # 5. 각 항목을 일정 간격(y 좌표)으로 표 형태로 렌더링
    start_y = 150
    line_spacing = 55

    for idx, (label, val) in enumerate(items):
        current_y = start_y + (idx * line_spacing)

        # 항목 라벨 표시 (좌측)
        draw.text((60, current_y), f"• {label}", fill=(0, 0, 0), font=font_label)

        # 실제 값 표시 (우측)
        draw.text(
            (210, current_y), f":  {val}", fill=(0, 0, 0), font=font_content
        )

        # 구분선 추가
        draw.line(
            [(50, current_y + 35), (width - 50, current_y + 35)],
            fill=(220, 220, 220),
            width=1,
        )

    # 6. 하단 직인/발급기관 안내 문구 추가
    draw.text(
        (160, height - 120),
        "국 세 청 장  [직인생략]",
        fill=(0, 0, 0),
        font=font_title,
    )

    # 7. 지정된 파일명으로 이미지 저장
    image.save(file_name, "JPEG")
    print(f"✅ 테스트용 사업자등록증 이미지 생성 완료: {file_name}")


if __name__ == "__main__":
    create_business_registration_image()

✅ 테스트용 사업자등록증 이미지 생성 완료: business_registration.jpg


In [7]:
# OpenCV 라이브러리를 불러옵니다 (이미지 읽기, 그레이스케일 변환, 노이즈 제거, 이진화 등 전처리용).
import cv2

# EasyOCR 라이브러리를 불러옵니다 (딥러닝 기반 텍스트 감지 및 인식 모델 사용).
import easyocr

# 정규표현식(Regular Expression) 모듈을 불러옵니다 (인식된 문자열에서 사업자번호, 전화번호 등 패턴 추출용).
import re

# 행렬 연산 및 배열 처리를 위한 NumPy 라이브러리를 불러옵니다.
import numpy as np


# ==========================================
# 1. 이미지 전처리 (Pre-processing)
# ==========================================
def preprocess_image(image_path):
    # 지정한 파일 경로에서 이미지를 BGR 칼라 포맷 형태로 읽어옵니다.
    img = cv2.imread(image_path)
    
    # 이미지가 존재하지 않거나 경로가 잘못되었을 경우 에러를 발생시킵니다.
    if img is None:
        raise FileNotFoundError(f"이미지를 찾을 수 없습니다: {image_path}")

    # 1-1. 컬러 이미지를 처리 속도 향상 및 채널 축소를 위해 흑백(Grayscale) 이미지로 변환합니다.
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # 1-2. 가우시안 블러(Gaussian Blur)를 적용하여 이미지의 자잘한 노이즈와 경계면을 부드럽게 완화시킵니다.
    # - (3, 3): 커널(필터) 크기, 0: 표준편차 자동 계산
    blurred = cv2.GaussianBlur(gray, (3, 3), 0)

    # 1-3. 적응형 이진화(Adaptive Thresholding)를 수행하여 문자를 흑/백 명확한 이진 이미지로 만듭니다.
    # - 조명이 불균일하거나 그림자가 진 이미지에서도 영역별로 문자를 선명하게 만들어 인식률을 높입니다.
    # - 255: 최대 임계값, cv2.ADAPTIVE_THRESH_GAUSSIAN_C: 가우시안 윈도우 기반 알고리즘 사용
    # - cv2.THRESH_BINARY: 임계값 초과는 픽셀을 흰색, 미만은 검은색으로 처리
    # - 11: 영역 계산을 위한 블록 크기, 2: 계산된 임계값에서 차감할 상수 C
    binary = cv2.adaptiveThreshold(
        blurred, 255, 
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
        cv2.THRESH_BINARY, 15, 2
    )

    # 원본 이미지와 전처리가 완료된 이진화 이미지를 함께 반환합니다.
    return img, binary


# ==========================================
# 2. OCR 모델 인식 (Text Detection & Recognition)
# ==========================================
def run_ocr(image, gpu=False):
    # EasyOCR의 Reader 객체를 초기화합니다.
    # - ['ko', 'en']: 인식 언어로 한국어와 영어를 설정합니다.
    # - gpu: GPU 사용 여부를 지정합니다 (기본값은 CPU 모드인 False).
    reader = easyocr.Reader(['ko', 'en'], gpu=gpu)
    
    # 전처리된 이미지에서 텍스트 영역을 찾고 문자를 읽어옵니다.
    # 반환 형식: [(영역 좌표 List), '인식된 텍스트', 신뢰도 Float] 구조의 튜플 리스트
    results = reader.readtext(image)
    return results


# ==========================================
# 3. 후처리 및 데이터 구조화 (Post-processing)
# ==========================================
def parse_ocr_results(ocr_results, confidence_threshold=0.4):
    # 신뢰도 기준을 통과한 순수 텍스트만 모아둘 리스트를 생성합니다.
    full_text_list = []
    
    # OCR 결과 리스트를 한 줄씩 순회합니다.
    for bbox, text, prob in ocr_results:
        # 모델의 인식 신뢰도(prob)가 설정한 기준값(기본 0.4 / 40%) 이상인 경우에만 채택합니다.
        if prob >= confidence_threshold:
            # 텍스트 앞뒤의 불필요한 공백 문자(스페이스, 탭 등)를 제거하여 리스트에 추가합니다.
            full_text_list.append(text.strip())
            
    # 추출된 개별 텍스트 줄들을 줄바꿈문자(\n)로 연결하여 하나의 문자열로 합칩니다.
    full_text = "\n".join(full_text_list)

    # 텍스트 내에서 정규표현식으로 추출할 키-값(Key-Value) 형태의 결과 사전(Dictionary) 구조를 정의합니다.
    extracted_data = {
        "business_number": None,  # 사업자등록번호 패턴 (예: 000-00-00000)
        "phone_number": None,     # 전화번호 패턴 (예: 02-123-4567, 010-1234-5678)
        "raw_text": full_text     # 필터링을 거친 전체 텍스트
    }

    # 정규표현식 매칭 패턴을 미리 컴파일합니다.
    # biz_pattern: 숫자 3자리 - 숫자 2자리 - 숫자 5자리 (사업자등록번호)
    biz_pattern = re.compile(r'\d{3}-\d{2}-\d{5}')
    # phone_pattern: 0으로 시작하는 2~3자리 - 숫자 3~4자리 - 숫자 4자리 (일반전화/휴대폰번호)
    phone_pattern = re.compile(r'0\d{1,2}-\d{3,4}-\d{4}')

    # 전체 인식 텍스트 내에서 패턴에 부합하는 첫 번째 문자열을 검색합니다.
    biz_match = biz_pattern.search(full_text)
    phone_match = phone_pattern.search(full_text)

    # 사업자등록번호 패턴이 일치하면 결과 사전에 저장합니다.
    if biz_match:
        extracted_data["business_number"] = biz_match.group()
    # 전화번호 패턴이 일치하면 결과 사전에 저장합니다.
    if phone_match:
        extracted_data["phone_number"] = phone_match.group()

    # 구조화된 파싱 결과를 반환합니다.
    return extracted_data


# ==========================================
# 4. 전체 파이프라인 실행
# ==========================================
def ocr_pipeline(image_path):
    # 1단계: 이미지 전처리 진행
    print("[1/3] 이미지 전처리 진행 중...")
    original_img, preprocessed_img = preprocess_image(image_path)

    # 2단계: 전처리된 이미지로 OCR 실행
    print("[2/3] OCR 텍스트 인식 진행 중...")
    # 전처리 완료된 이진화 이미지(preprocessed_img)를 OCR 입력으로 전송합니다.
    ocr_results = run_ocr(preprocessed_img)

    # 3단계: 인식 결과에서 주요 키 정보 추출 및 구조화
    print("[3/3] 데이터 추출 및 구조화 중...")
    structured_data = parse_ocr_results(ocr_results)

    # 최종 추출 결과 반환
    return structured_data


# --- 메인 모듈 직접 실행 시 구동할 메인 함수 ---
if __name__ == "__main__":
    # 파이프라인에 입력할 테스트 이미지 파일명 지정
    sample_image = "business_registration.jpg"
    
   
    # ocr_pipeline() 함수를 실행하여 구조화 데이터를 수집합니다.
    result = ocr_pipeline(sample_image)
    
    # 추출 결과를 콘솔 화면에 가독성 좋게 출력합니다.
    print("\n=== 최종 구조화 추출 결과 ===")
    print(f"사업자번호: {result['business_number']}")
    print(f"전화번호: {result['phone_number']}")
    print(f"\n전체 인식 텍스트:\n{result['raw_text']}")

Using CPU. Note: This module is much faster with a GPU.


[1/3] 이미지 전처리 진행 중...
[2/3] OCR 텍스트 인식 진행 중...
[3/3] 데이터 추출 및 구조화 중...

=== 최종 구조화 추출 결과 ===
사업자번호: 123-45-67890
전화번호: 02-1233-5678

전체 인식 텍스트:
사업자등록증
123-45-67890
대표자평
사업장 소재지
대표 전화번호
02-1233-5678
사업의 종류
국 세 청 장 [직인생락]


In [8]:
import cv2
import numpy as np
from PIL import Image, ImageDraw, ImageFont

def create_advanced_receipt(output_path="receipt_sample.jpg"):
    # 1. 캔버스 생성 (흰색 배경, 450x650)
    canvas = Image.new('RGB', (450, 650), color=(255, 255, 255))
    draw = ImageDraw.Draw(canvas)

    # 2. 폰트 로드 (OS별 나눔고딕/맑은고딕 등 한글 폰트 설정)
    try:
        # 윈도우 기본 폰트
        font_regular = ImageFont.truetype("malgun.ttf", 15)
        font_bold = ImageFont.truetype("malgunbd.ttf", 18)
    except:
        try:
            # macOS 기본 폰트
            font_regular = ImageFont.truetype("/System/Library/Fonts/Supplemental/AppleGothic.ttf", 15)
            font_bold = ImageFont.truetype("/System/Library/Fonts/Supplemental/AppleGothic.ttf", 18)
        except:
            # 폰트 로드 실패 시 기본 폰트 (한글이 깨질 수 있음)
            font_regular = ImageFont.load_default()
            font_bold = font_regular

    # 3. 영수증 텍스트 구성 (정규식 패턴 매칭 항목 포함)
    lines = [
        ("[ AI 스마트 스토어 ]", font_bold, 130),
        ("------------------------------------------", font_regular, 30),
        ("사업자번호: 214-88-12345", font_regular, 30),
        ("전화번호: 02-555-7890", font_regular, 30),
        ("주소: 서울시 강남구 테헤란로 456", font_regular, 30),
        ("------------------------------------------", font_regular, 30),
        ("상품명              수량          금액", font_regular, 30),
        ("------------------------------------------", font_regular, 30),
        ("딥러닝 교재           1        35,000", font_regular, 30),
        ("아메리카노           2         9,000", font_regular, 30),
        ("------------------------------------------", font_regular, 30),
        ("합계금액                     44,000", font_bold, 30),
        ("==========================================", font_regular, 30),
        ("거래일자: 2026-08-27", font_regular, 30),
        ("이용해 주셔서 감사합니다.", font_regular, 120)
    ]

    # 4. 텍스트 그리기
    y_offset = 30
    for text, font, x_offset in lines:
        draw.text((x_offset, y_offset), text, fill=(0, 0, 0), font=font)
        y_offset += 35

    # 5. [실무 테스트용] 이미지를 OpenCV로 변환 후 약 -2도 기울이기 (Deskew 기능 검증)
    cv_img = cv2.cvtColor(np.array(canvas), cv2.COLOR_RGB2BGR)
    (h, w) = cv_img.shape[:2]
    center = (w // 2, h // 2)
    
    # -2.5도 회전 행렬 생성
    M = cv2.getRotationMatrix2D(center, -2.5, 1.0)
    rotated_img = cv2.warpAffine(
        cv_img, M, (w, h), 
        flags=cv2.INTER_CUBIC, 
        borderMode=cv2.BORDER_CONSTANT, 
        borderValue=(255, 255, 255)
    )

    # 6. 이미지 저장
    cv2.imwrite(output_path, rotated_img)
    print(f"테스트용 영수증 이미지 파일이 생성되었습니다: {output_path}")

if __name__ == "__main__":
    create_advanced_receipt()

테스트용 영수증 이미지 파일이 생성되었습니다: receipt_sample.jpg


In [ ]:
# OpenCV 라이브러리: 이미지 로드, 색상 변환, 회전 행렬 생성 및 아핀 변환에 사용됩니다.
import cv2

# NumPy 라이브러리: 이미지의 픽셀 좌표를 다차원 배열(Matrix)로 다루고 추출할 때 사용됩니다.
import numpy as np

# 이미지의 기울어진 각도를 자동 측정하여 수평으로 보정하는 단독 함수 정의
def deskew_image(image_path):
    # --------------------------------------------------------------------------
    # 1. 테스트할 이미지를 파일 경로에서 BGR 컬러 포맷으로 읽어옵니다.
    # --------------------------------------------------------------------------
    image = cv2.imread(image_path)

    # 이미지 파일이 존재하지 않거나 경로가 잘못되었을 경우 예외를 발생시킵니다.
    if image is None:
        raise FileNotFoundError(f"이미지를 로드할 수 없습니다: {image_path}")

    # --------------------------------------------------------------------------
    # 2. 기울기 계산을 위해 BGR 컬러 이미지를 흑백(Grayscale) 이미지로 변환합니다.
    # --------------------------------------------------------------------------
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # --------------------------------------------------------------------------
    # 3. 픽셀 값이 255(흰색) 미만인 어두운 영역(텍스트 및 객체)의 좌표를 모두 추출합니다.
    # - 255: Grayscale에서 가장 밝은 흰색(Pure White) 값
    # - gray < 255: 글자나 객체가 존재하는 어두운 픽셀 영역들을 필터링
    # - np.where: 조건을 만족하는 픽셀의 (y, x) 인덱스 위치 반환
    # - np.column_stack: y, x 좌표 배열을 2D 좌표 (y, x) 행렬 형태로 결합
    # --------------------------------------------------------------------------
    coords = np.column_stack(np.where(gray < 255))

    # 추출된 픽셀 좌표가 0개인 경우(빈 바탕 이미지) 보정 없이 원본 이미지와 0도 각도를 반환합니다.
    if coords.shape[0] == 0:
        print("유효한 텍스트/객체 픽셀을 찾지 못했습니다.")
        return image, 0.0

    # --------------------------------------------------------------------------
    # 4. 텍스트 픽셀 좌표들을 둘러싸는 최소 크기의 회전된 사각형(Bounding Box)을 계산합니다.
    # - cv2.minAreaRect: 픽셀 좌표 집합을 감싸는 최소 영역 사각형 생성
    # - rect 반환 구조: (중심점(x,y), (가로,세로 크기), 회전 각도)
    # - rect[-1]: 반환 튜플의 마지막 요소인 회전 각도를 가져옴 (범위: -90도 ~ 0도)
    # --------------------------------------------------------------------------
    rect = cv2.minAreaRect(coords)
    angle = rect[-1]

    # --------------------------------------------------------------------------
    # 5. minAreaRect 함수가 반환하는 각도의 범위(-90도 ~ 0도)를 실제 보정용 각도로 정규화합니다.
    # - -45도 미만: 오른쪽으로 가파르게 기울어진 상태로 판단하여 (90 + angle)의 음수값 적용
    # - -45도 이상: 일반적인 기울기 상태로 판단하여 각도 부호 반전
    # --------------------------------------------------------------------------
    if angle < -45:
        angle = -(90 + angle)
    else:
        angle = -angle

    # 콘솔에 측정된 기울기 각도를 소수점 둘째 자리까지 출력합니다.
    print(f"[측정된 기울기 각도]: {angle:.2f}도")

    # --------------------------------------------------------------------------
    # 6. 절대 각도가 1.0도 미만이면 회전에 따른 이미지 화질 저하를 막기 위해 보정을 생략합니다.
    # - 1.0: 임계 각도 (1도 미만의 미세한 기울기는 보정하지 않음)
    # --------------------------------------------------------------------------
    if abs(angle) < 1.0:
        print("기울기가 1도 미만이므로 보정을 생략합니다.")
        return image, angle

    # --------------------------------------------------------------------------
    # 7. 이미지의 중심점을 구하고, 회전 변환 행렬 M을 생성합니다.
    # - h: 이미지 세로 높이, w: 이미지 가로 너비 (shape[:2]로 추출)
    # - // 2: 정수 나눗셈 연산자로 가로/세로 중심점 좌표 계산
    # - cv2.getRotationMatrix2D(center, angle, scale):
    #   * center: 회전 중심 좌표
    #   * angle: 회전 각도 (양수: 시계 반대 방향, 음수: 시계 방향)
    #   * 1.0: 확대/축소 배율 (100% 원본 비율 유지)
    # --------------------------------------------------------------------------
    (h, w) = image.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)

    # --------------------------------------------------------------------------
    # 8. 아핀 변환(Affine Transformation)을 적용하여 이미지를 실제로 회전 보정합니다.
    # - flags=cv2.INTER_CUBIC: 고품질 3차 스플라인 보간법 사용 (회전 시 글자 경계 유지)
    # - borderMode=cv2.BORDER_REPLICATE: 회전 후 생기는 외곽 여백을 가장자리 픽셀 값으로 복제하여 채움
    # --------------------------------------------------------------------------
    rotated = cv2.warpAffine(
        image, M, (w, h), 
        flags=cv2.INTER_CUBIC, 
        borderMode=cv2.BORDER_REPLICATE
    )

    # 보정된 이미지와 측정된 각도를 반환합니다.
    return rotated, angle


# ==========================================
# 단독 테스트 실행 영역 (Main Routine)
# ==========================================
# 이 파일이 파이썬에서 직접 실행될 때 아래 코드를 수행합니다.
if __name__ == "__main__":
    # 테스트할 기울어진 입력 이미지 경로 지정
    test_image_path = "receipt_sample.jpg"

    try:
        # 기울기 보정 함수 호출 (보정된 이미지와 측정 각도를 리턴받음)
        corrected_img, detected_angle = deskew_image(test_image_path)

        # 원본 이미지와 비교 창을 띄우기 위해 원본 이미지를 다시 로드합니다.
        original_img = cv2.imread(test_image_path)

        # 보정 완료된 결과 이미지를 파일로 저장합니다.
        output_path = "deskewed_result.jpg"
        cv2.imwrite(output_path, corrected_img)
        print(f"보정된 이미지가 성공적으로 저장되었습니다: {output_path}")

        # OpenCV 창을 띄워 원본 이미지와 보정된 이미지를 시각적으로 비교합니다.
        cv2.imshow("1. Original Image (Rotated)", original_img)
        cv2.imshow("2. Deskewed Result", corrected_img)

        # 키 입력이 들어올 때까지 창을 닫지 않고 대기합니다. (0: 무한 대기)
        cv2.waitKey(0)

        # 열려있는 모든 OpenCV 시각화 창을 닫아 메모리를 해제합니다.
        cv2.destroyAllWindows()

    # 이미지 파일 읽기 실패 등의 예외 발생 시 에러 메시지 출력
    except Exception as e:
        print(f"[오류 발생]: {e}")


[측정된 기울기 각도]: -0.00도
기울기가 1도 미만이므로 보정을 생략합니다.
보정된 이미지가 성공적으로 저장되었습니다: deskewed_result.jpg
